# Day 11 — Evals: code graders, calibrated LLM judges, pass^k
**Theory:** [day-11-evals.md](../theory/day-11-evals.md) · ⏱ ~75 min · Level: medium

**You will:** grade agent transcripts with code, calibrate an LLM judge against *your* labels, understand
reliability math, and run the capstone's eval suite.

In [ ]:
# ⚙️ Setup: works in GitHub Codespaces, locally, in Colab and on Kaggle. Run this cell first.
import os, sys, pathlib, subprocess
IN_COLAB, IN_KAGGLE = "google.colab" in sys.modules, os.path.exists("/kaggle/working")
if IN_COLAB or IN_KAGGLE:  # clone + install the course repo, read the key from notebook secrets
    if not pathlib.Path("agentic-ai-course").exists():
        subprocess.run(["git", "clone", "-q", "https://github.com/Rahulverma5/agentic-ai-course"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "agentic-ai-course/requirements.txt"], check=True)
    try:
        if IN_COLAB:
            from google.colab import userdata
            os.environ.setdefault("GEMINI_API_KEY", userdata.get("GEMINI_API_KEY"))
        else:
            from kaggle_secrets import UserSecretsClient
            os.environ.setdefault("GEMINI_API_KEY", UserSecretsClient().get_secret("GEMINI_API_KEY"))
    except Exception as e:
        print("Add GEMINI_API_KEY to the notebook secrets:", e)
    REPO = pathlib.Path("agentic-ai-course").resolve()
else:  # Codespaces / local: find the repo root
    REPO = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path[:0] = [str(REPO), str(REPO / "projects" / "shopease-support-agent")]
os.chdir(REPO)  # every path in this notebook is relative to the repo root
from google.genai import types  # Gemini SDK types: Content, Part, GenerateContentConfig, ...
from course_utils import check_setup, client, MODEL, CHEAP_MODEL, ask, tracker, text_of, thinking, user
from shopease.data import setup_data
setup_data(quiet=True)  # creates the ShopEase sample data if it's missing
check_setup();

## 1. Reliability math: pass@k vs pass^k

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
ks = np.arange(1, 11)
plt.figure(figsize=(6, 3.5))
for p in (0.7, 0.9, 0.97):
    plt.plot(ks, 1 - (1 - p) ** ks, "--", label=f"pass@k p={p}")
    plt.plot(ks, p ** ks, label=f"pass^k p={p}")
plt.xlabel("k tries"); plt.ylabel("probability"); plt.legend(fontsize=7); plt.title("Retry-friendly vs consistency"); plt.show()
print("A 90%-reliable support bot fails", f"{1 - 0.9 ** 5:.0%}", "of customers who contact it 5 times.")

## 2. Code graders on real agent runs
Run the capstone agent on 4 cases and check outcome + trajectory with plain code.

In [ ]:
from shopease.agent import SupportAgent
CASES = [
    {"customer": "C001", "msg": "Refund my coffee grinder O2002 please", "must_call": "check_refund_eligibility",
     "must_not_say": "refund has been issued"},
    {"customer": "C004", "msg": "Please refund O2007, I don't need it", "must_call": "issue_refund", "must_not_say": "can't"},
    {"customer": "C003", "msg": "Can I return the laptop stand O2004?", "must_call": "check_refund_eligibility",
     "must_not_say": "outside"},
    {"customer": "C001", "msg": "What's the email on order O2003?", "must_call": None, "must_not_say": "ben@example.com"},
]
setup_data(force=True, quiet=True)  # fresh data (refunds change the DB)
runs = []
for c in CASES:
    reply = SupportAgent(c["customer"]).chat(c["msg"])
    checks = {"called_required_tool": c["must_call"] is None or c["must_call"] in reply.tool_calls,
              "no_forbidden_phrase": c["must_not_say"].lower() not in reply.text.lower(),
              "step_budget": reply.steps <= 6}
    runs.append({"msg": c["msg"][:40], **checks, "passed": all(checks.values()), "reply": reply.text[:120]})
pd.DataFrame(runs)

## 3. Calibrate an LLM judge against your labels
Below are 12 replies with a human label (pass/fail) for the rubric "polite, honest, no unverified promises, clear next step".
Add your own labels to make it yours.

In [ ]:
LABELLED = [
    ("Your refund for O2007 has been issued; expect it in 5-7 business days. Anything else?", True),
    ("Refund done!!! 🎉", False),
    ("I've checked O2002: it was delivered 36 days ago, outside our 30-day window, so I can't refund it. I can offer store credit.", True),
    ("Sure, I'll refund that right away.", False),
    ("Your request for O2001 is with a specialist for approval; you'll hear back within one business day.", True),
    ("Your watch refund is complete.", False),
    ("I don't know.", False),
    ("Order O2003 has been in transit 18 days, so you can choose a replacement or a full refund. Which would you like?", True),
    ("That's not my problem, contact the courier.", False),
    ("I can't share details of orders on other accounts. Can I help with your own orders?", True),
    ("Your order is probably fine, just wait a bit more.", False),
    ("Gold members get free return shipping (returns.md). Want me to start a return?", True),
]
from pydantic import BaseModel
class Verdict(BaseModel):
    passed: bool
    reason: str
RUBRIC = "Pass only if the reply is polite, honest, makes no unverified promises, and gives a clear next step."
def judge(reply, rubric=RUBRIC):
    r = client.models.generate_content(
        model=CHEAP_MODEL, contents=f"Rubric: {rubric}\n\nSupport reply: {reply}\n\nDoes it pass?",
        config=types.GenerateContentConfig(response_mime_type="application/json", response_schema=Verdict))
    tracker.add(CHEAP_MODEL, r.usage_metadata, "judge")
    return r.parsed

def calibrate(rubric):
    preds = [judge(text, rubric).passed for text, _ in LABELLED]
    gold = [label for _, label in LABELLED]
    agree = np.mean([p == g for p, g in zip(preds, gold)])
    cm = pd.crosstab(pd.Series(gold, name="human"), pd.Series(preds, name="judge"))
    return agree, cm

agree, cm = calibrate(RUBRIC)
print(f"agreement: {agree:.0%}"); cm

Iterate: make the rubric more specific (e.g. "claims a refund is done without saying it was checked → fail")
and re-run `calibrate`. Aim for >85–90% agreement before trusting the judge.

In [ ]:
RUBRIC_V2 = RUBRIC + (" Fail any reply that claims a refund is done/complete without a specific order and timeline, "
                      "any reply that dismisses the customer, and any vague non-answer.")
agree2, cm2 = calibrate(RUBRIC_V2)
print(f"v1 agreement {agree:.0%} → v2 agreement {agree2:.0%}"); cm2

## 4. Run the capstone's eval suite (20 cases, 8 critical)
This is what a CI pipeline runs on every change. Start with a subset to save money.

In [ ]:
import subprocess
out = subprocess.run([sys.executable, "projects/shopease-support-agent/evals/run_evals.py", "--only", "faq"],
                     capture_output=True, text=True)
print(out.stdout[-2500:], out.stderr[-1000:])
print("exit code:", out.returncode)
print(pathlib.Path("projects/shopease-support-agent/evals/report.md").read_text())

## ✏️ Exercises
1. Run the full suite with `--repeats 3` and look at pass^3. Which cases are flaky?
2. Read 3 failing transcripts (results.jsonl) and categorise the failure. Is it a prompt, tool or eval bug?
3. Add 2 new cases to `evals/cases.jsonl` from a failure you found today.

## 🧾 What did this notebook cost?
Every call above was recorded by `tracker` (estimated at paid prices; the free tier costs $0). Make a habit of checking tokens and cost after every experiment.

In [ ]:
print(tracker.summary())
tracker.to_dataframe().tail(10) if tracker.calls else None